In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from collections import defaultdict
import os

fifa_ranking_2022 = pd.read_csv('fifa_ranking_2022-10-06.csv')

fifa_ranking_2026 = pd.read_csv('fifa_ranking_2026-06-08.csv')

matches = pd.read_csv('matches_1930_2022.csv')

schedule_2026 = pd.read_csv('schedule_2026.csv')

world_cup = pd.read_csv('world_cup.csv')

print("Successfully loaded all datasets.")

from sklearn.linear_model import LogisticRegression
import inspect

print(inspect.signature(LogisticRegression))


In [ ]:
# Display the first few rows of each dataset
print("FIFA Ranking 2022:")
print(fifa_ranking_2022.head().to_string(index=False))

print("\nFIFA Ranking 2026:")
print(fifa_ranking_2026.head().to_string(index=False))

print("\nMatches 1930-2022:")
print(matches.head().to_string(index=False))

print("\nMatches 2026:")
print(schedule_2026.head().to_string(index=False))

print("\nWorld Cup History:")
print(world_cup.head().to_string(index=False))

In [ ]:
schedule_2026_missing_val = schedule_2026.isnull().sum()
fifa_ranking_2022_missing_val = fifa_ranking_2022.isnull().sum()
fifa_ranking_2026_missing_val = fifa_ranking_2026.isnull().sum()
matches_missing_val = matches.isnull().sum()
world_cup_missing_val = matches.isnull().sum()
missing_total = schedule_2026_missing_val + fifa_ranking_2022_missing_val + fifa_ranking_2026_missing_val + matches_missing_val + world_cup_missing_val

print("Display missing values in all data files.")
print(missing_total)

# Filling missing values with 0
schedule_2026 = schedule_2026.fillna(0)
fifa_ranking_2022 = fifa_ranking_2022.fillna(0)
fifa_ranking_2026 = fifa_ranking_2026.fillna(0)
matches = matches.fillna(0)
world_cup = world_cup.fillna(0)


team_mapping = {
    "United States" : "USA",
    "Bosnia-Herzegovina" : "Bosnia and Herzegovina",
    "Congo DR" : "Congo"
}

schedule_2026['home_team'] = schedule_2026["home_team"].replace(team_mapping)
schedule_2026['away_team'] = schedule_2026["away_team"].replace(team_mapping)
matches['home_team'] = matches["home_team"].replace(team_mapping)
matches['away_team'] = matches["away_team"].replace(team_mapping)


In [ ]:
merge_ranking = pd.merge(fifa_ranking_2026, fifa_ranking_2022, on='team', suffixes=('_2026', '_2022'))

merge_ranking['rank_change'] = merge_ranking['rank_2022'] - merge_ranking['rank_2026']

# Display the merged ranking
print(merge_ranking[['team', 'rank_2022', 'rank_2026', 'rank_change']].head().to_string(index=False))

In [ ]:
# World Cup summary
print('World Cup Data Head.')
print(world_cup.head().to_string(index=False))

# Display Championship
champion_table = world_cup.groupby('Champion')['Year'].apply(lambda x: sorted(x))
champion_table = champion_table.reset_index()
champion_table['Count'] = champion_table['Year'].apply(len)
champion_table = champion_table.sort_values('Count', ascending=False)[['Champion', 'Count', 'Year']]
champion_table = champion_table.reset_index(drop=True)

print("\nWorld Cup Champions Frequency")
print(champion_table)

champion_counts = world_cup['Champion'].value_counts()
plt.figure(figsize=(12, 6))
sns.barplot(x=champion_counts.index, y=champion_counts.values, hue=champion_counts.index, palette='viridis', legend=False)
plt.title('World Cup Champions Frequency')
plt.xticks(rotation=45)
plt.xlabel('Champion')
plt.ylabel('Cups')
plt.show()

In [ ]:
matches['Date'] = pd.to_datetime(matches['Date'], errors='coerce')
matches = matches.sort_values('Date').reset_index(drop=True)
print("Display match dataset Date column info, after modifying.")
print(matches['Date'].describe())

# Display mathces per year
matches_per_year = matches.groupby('Year').size()
plt.figure(figsize=(12,6))
sns.lineplot(x=matches_per_year.index, y=matches_per_year.values, marker='o')
plt.title('Numer of matches per year')
plt.xlabel('Yead')
plt.ylabel('Number of matches')
plt.show()

In [ ]:
schedule_2026['Date'] = pd.to_datetime(schedule_2026['Date'], errors='coerce')
print("Date schedules 2026 after modifying.")
print(schedule_2026['Date'].describe())

print("\nSchedule 2026 head.")
print(schedule_2026.head().to_string(index=False))

plt.figure(figsize=(12,6))
sns.countplot(data=schedule_2026, x='Date', palette='Set2', order=sorted(schedule_2026['Date'].unique()), hue='Date', legend=False)
plt.title("Number of Matches per Day - Tournament Schedule 2026")
plt.xlabel('Date')
plt.ylabel('Number of matches')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
def get_result(row):
    if row['home_score'] > row['away_score']:
        return "Home" # Home team win
    elif row['home_score'] < row['away_score']:
        return "Away" # Away team win
    else:
        return "Draw" # Draw

matches['result'] = matches.apply(get_result, axis=1)

home_rankings = fifa_ranking_2026[[
    "team",
    "rank",
    "points"
]].rename(columns={
    "team": "home_team",
    "rank": "home_rank",
    "points": "home_points"
})

away_rankings = fifa_ranking_2026[[
    "team",
    "rank",
    "points"
]].rename(columns={
    "team": "away_team",
    "rank": "away_rank",
    "points": "away_points"
})

matches = matches.merge(
    home_rankings,
    on='home_team',
    how='left'
)

matches = matches.merge(
    away_rankings,
    on='away_team',
    how='left'
)

matches['rank_diff'] = (
    matches['home_rank']
    - matches['away_rank']
)

matches['points_diff'] = (
    matches['home_points']
    - matches['away_points']
)

matches['strength_gap'] = abs(
    matches['home_points'] - matches['away_points']
)

matches['elo_prob_home'] = 1 / (
    1 + 10 ** ((matches['away_points'] - matches['home_points']) / 400)
)

matches['rank_gap_abs'] = abs(matches['rank_diff'])

matches['points_gap_abs'] = abs(matches['points_diff'])

matches = matches.fillna(0)

print(matches[[
    "home_team",
    "away_team",
    "home_rank",
    "away_rank",
    "home_score",
    "away_score", 
    "home_points",
    "away_points",
    "result",
    "rank_diff",
    "points_diff",
    "strength_gap",
    "elo_prob_home",
    "rank_gap_abs",
    "points_gap_abs"
]].head(200).to_string(index=False))

In [ ]:
team_history = defaultdict(list)

matches['home_last5_win_rate'] = 0.0
matches['away_last5_win_rate'] = 0.0

matches['home_last5_goals_for'] = 0.0
matches['away_last5_goals_for'] = 0.0

matches['home_last5_goals_against'] = 0.0
matches['away_last5_goals_against'] = 0.0

for idx, row in matches.iterrows():

    home = row['home_team']
    away = row['away_team']

    home_history = team_history[home][-5:]
    away_history = team_history[away][-5:]

    if home_history:

        matches.loc[idx, 'home_last5_win_rate'] = np.mean(
            [m['win'] for m in home_history]
        )

        matches.loc[idx, 'home_last5_goals_for'] = np.mean(
            [m['goals_for'] for m in home_history]
        )

        matches.loc[idx, 'home_last5_goals_against'] = np.mean(
            [m['goals_against'] for m in home_history]
        )
    
    if away_history:

        matches.loc[idx, 'away_last5_win_rate'] = np.mean(
            [m['win'] for m in away_history]
        )

        matches.loc[idx, 'away_last5_goals_for'] = np.mean(
            [m['goals_for'] for m in away_history]
        )

        matches.loc[idx, 'away_last5_goals_against'] = np.mean(
            [m['goals_against'] for m in away_history]
        )
home_win = int(row['home_score'] > row['away_score'])
away_win = int(row['away_score'] > row['home_score'])
team_history[home].append({
    'win': home_win,
    'goals_for': row['home_score'],
    'goals_against': row['away_score']
})

team_history[away].append({
    'win': away_win,
    'goals_for': row['away_score'],
    'goals_against': row['home_score']
 })

matches['form_diff'] = (
    matches['home_last5_win_rate']
    - matches['away_last5_win_rate']
)

matches['attack_diff'] = (
    matches['home_last5_goals_for']
    - matches['away_last5_goals_for']
)

matches['defense_diff'] = (
    matches['away_last5_goals_against']
    - matches['home_last5_goals_against']
)

In [ ]:
feature = [
    'rank_diff',
    'points_diff',
    'strength_gap',
    'elo_prob_home',
    'rank_gap_abs',
    'points_gap_abs',

    'home_last5_win_rate',
    'away_last5_win_rate',

    'home_last5_goals_for',
    'away_last5_goals_for',

    'home_last5_goals_against',
    'away_last5_goals_against',

    'form_diff',
    'attack_diff',
    'defense_diff'
]

X = matches[feature]
y = matches['result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(
    max_iter=3000,
    class_weight='balanced',
    random_state=42
)
model.fit(X_train, y_train)

model.fit(X_train,
    y_train
)

prediction = model.predict(X_test)
accuracy_score = accuracy_score(y_test, prediction)

print(prediction)
print(accuracy_score)

print(matches['result'].value_counts(normalize=True))

In [ ]:
matches_2026 = schedule_2026.merge(
    home_rankings,
    on="home_team",
    how="left"
)

matches_2026 = matches_2026.merge(
    away_rankings,
    on="away_team",
    how="left"
)

matches_2026['rank_diff'] = (
    matches_2026['home_rank']
    - matches_2026['away_rank']
)

matches_2026['points_diff'] = (
    matches_2026['home_points']
    - matches_2026['away_points']
)

matches_2026['strength_gap'] = abs(
    matches_2026['home_points'] - matches_2026['away_points']
)

matches_2026['elo_prob_home'] = 1 / (
    1 + 10 ** ((matches_2026['away_points'] - matches_2026['home_points']) / 400)
)

matches_2026['rank_gap_abs'] = abs(matches_2026['rank_diff'])

matches_2026['points_gap_abs'] = abs(matches_2026['points_diff'])

matches_2026['home_team'] = matches_2026["home_team"].replace(team_mapping)
matches_2026['away_team'] = matches_2026["away_team"].replace(team_mapping)
matches_2026 = matches_2026.fillna(0)

X_2026 = matches_2026[
    ['rank_diff',
     'points_diff',
     'strength_gap',
     'elo_prob_home',
     'rank_gap_abs',
     'points_gap_abs',
     'home_last5_win_rate',
     'away_last5_win_rate',
     'home_last5_goals_for',
     'away_last5_goals_for',
     'home_last5_goals_against',
     'away_last5_goals_against',
     'form_diff',
     'attack_diff',
     'defense_diff']
]

matches_2026['prediction'] = (
    model.predict(X_2026)
)

probs = model.predict_proba(
    X_2026
)
print(matches_2026['prediction'].value_counts(normalize=True))
print(model.classes_) # ['Away', 'Draw', 'Home]
print(probs)
print(matches_2026[[
    "home_team",
    "away_team",
    "home_rank",
    "away_rank",
    "rank_diff",
    "home_points",
    "away_points",
    "points_diff",
    "prediction"
]].head().to_string(index=False))


In [ ]:
print(confusion_matrix(y_test, prediction))
print(pd.crosstab(
    y_test,
    prediction,
    rownames=['Actual'],
    colnames=['Predicted']
))

In [ ]:
modelForest = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced'
)
modelForest.fit(X_train, y_train)

predictions = modelForest.predict(X_test)

probsForest = modelForest.predict_proba(X_2026)

print(matches_2026['prediction'].value_counts(normalize=True))
print(modelForest.classes_) # ['Away', 'Draw', 'Home']
print(probsForest)
print(matches_2026[[
    "home_team",
    "away_team",
    "home_rank",
    "away_rank",
    "rank_diff",
    "home_points",
    "away_points",
    "points_diff",
    "prediction"
]].head().to_string(index=False))